<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/csr-ranking.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · CSR-Style School Rankings

Reproduce a Competitive Strength Ranking (CSR): score each school by its
best finishes across a season, weighting each regatta by a **grade**.

The formula (from the ICSA CSR scoring method): a school's score in a
regatta is `base_points(place) × grade_multiplier(grade)`, where
`base_points` is `19 − place` (1st = 18 … 18th = 1, 0 beyond), and only a
school's **best** entry counts per regatta. Each school's season score is
the sum of its **top-N** regatta scores.

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## Optional: persist the cache on Colab

`scraper.load()` caches every fetched page under `./.scraper_cache`, so re-running a cell (or the whole notebook) after the first pass is instant. Colab wipes that folder on runtime reset — mount Drive and point the cache there first if you want it to survive. Set `SCRAPER_CACHE_DIR` **before** importing `scraper`:

```python
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.environ['SCRAPER_CACHE_DIR'] = '/content/drive/MyDrive/icsa_cache'
```

## The CSR scoring formula

Constants and the points function, straight from the CSR method.

In [ ]:
GRADE_MULTIPLIER = {0: 8.0, 1: 6.5, 2: 4.0, 3: 2.5, 4: 1.5, 5: 1.0}


# Base points: 1st = 18, 2nd = 17, ..., 18th = 1, 19th+ = 0
def base_points(place):
    return max(19 - place, 0)


# Grade 0 (national championships) uses a non-standard table
G0_SCORES = {
    1: 144,
    2: 140,
    3: 136,
    4: 132,
    5: 128,
    6: 124,
    7: 120,
    8: 116,
    9: 112,
    10: 108,
    11: 104,
    12: 100,
    13: 96,
    14: 92,
    15: 88,
    16: 84,
    17: 80,
    18: 76,
}


def csr_points(place, grade):
    if place < 1 or place > 18:
        return 0.0
    if grade == 0:
        return float(G0_SCORES.get(place, 0))
    return base_points(place) * GRADE_MULTIPLIER.get(grade, 1.0)


TOP_N = 6  # count each school's best 6 regattas (women's CSR uses 5)

## Grades

Each regatta has an ICSA-assigned **grade** 0–5 (0 = national championship,
highest weight; 5 = lowest). Grades are published by ICSA — plug them in by
regatta slug below. Anything not listed falls back to `DEFAULT_GRADE`.

> Set `DEFAULT_GRADE = None` to count **only** the regattas you grade
> explicitly (closest to official CSR, which scores a curated set). Leaving a
> numeric default grades every regatta — fine for a first look, but not the
> official result.

In [ ]:
SEASON = "s26"

# regatta_slug -> grade. Fill in the ICSA grades to reproduce official CSR.
GRADES = {
    "open-fleet-race-national-championships": 0,  # nationals
    # 'some-conference-championship': 1,
    # 'a-mid-tier-open': 3,
}
DEFAULT_GRADE = 5  # or None to score only graded regattas

## Scrape the season's fleet results

`sailors=False` skips the per-sailor pages this notebook doesn't need,
roughly halving the fetches; `progress=True` shows a per-regatta progress
bar while it runs. `.fleet()` narrows the dataset to fleet-scoring regattas
only. The first run scrapes the season (cached to disk); re-runs are
instant. CSR officially spans a full academic year (fall + spring) — pass
`['f25', 's26']` to `load` for that.

In [ ]:
import scraper

data = scraper.load(SEASON, sailors=False, progress=True).fleet()
df = data.results_frame()
print(len(data.regattas), "fleet regattas |", len(df), "school results")
df.head()

## Score each school

Best entry per school per regatta (A/B teams → best place), grade it, and
convert to CSR points.

In [ ]:
def grade_for(slug):
    return GRADES.get(slug, DEFAULT_GRADE)


# best (lowest) place per school per regatta
best = df.groupby(["regatta_slug", "school_slug", "school"], as_index=False)["place"].min()
best["grade"] = best["regatta_slug"].map(grade_for)
best = best[best["grade"].notna()].copy()  # drop ungraded when DEFAULT_GRADE is None
best["csr_points"] = [csr_points(int(p), int(g)) for p, g in zip(best["place"], best["grade"])]
best = best[best["csr_points"] > 0]
best.sort_values("csr_points", ascending=False).head()

## Rank the schools

Sum each school's **top-N** regatta scores.

In [ ]:
topn = (
    best.sort_values("csr_points", ascending=False).groupby(["school_slug", "school"]).head(TOP_N)
)

ranking = (
    topn.groupby(["school_slug", "school"], as_index=False)
    .agg(total_points=("csr_points", "sum"), counted=("csr_points", "size"))
    .sort_values("total_points", ascending=False)
    .reset_index(drop=True)
)
ranking.index = range(1, len(ranking) + 1)
ranking.round(1).head(25)

## Chart the top 20

In [ ]:
import matplotlib.pyplot as plt

top = ranking.head(20).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top["school"], top["total_points"])
plt.xlabel(f"CSR points (top {TOP_N} regattas)")
plt.title(f"{SEASON} CSR-style ranking")
plt.tight_layout()
plt.show()

---
**To reproduce official CSR:** fill `GRADES` with the ICSA-published grade
for each regatta (set `DEFAULT_GRADE = None`), and load the full academic
year (`scraper.load(['f25', 's26'], sailors=False)`). The women's ranking
uses `TOP_N = 5` and only women's regattas — filter with a grade set scoped
to those events.